# 03 - Point-in-time feature engineering

Features are built in SQL (`sql/features/01_loan_features.sql`) as of the day before each decision.

In [1]:
import duckdb
import pandas as pd

from credit_risk.config import get_settings

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.4f}".format)
settings = get_settings()
con = duckdb.connect(str(settings.paths.warehouse), read_only=True)
q = lambda sql: con.execute(sql).df()  # noqa: E731

In [2]:
from credit_risk.features.registry import FEATURES

pd.DataFrame(
    [(f.name, f.group.value, f.kind.value, f.description) for f in FEATURES],
    columns=["feature", "group", "kind", "description"],
).groupby("group").size()

group
application              7
borrower_profile         6
collections              4
delinquency              6
loan_history             8
recency                  3
repayment_behaviour      6
stability                4
transaction_behaviour    7
dtype: int64

## One repeat borrower's history, as the model saw it

In [3]:
loan = q("SELECT * FROM features.loan_features WHERE number_of_loans >= 3 ORDER BY loan_id LIMIT 1")
loan.T.head(40)

,0
loan_id,L0000005
borrower_id,B000002
decision_date,2025-06-04 00:00:00
as_of_date,2025-06-03 00:00:00
product_type,micro_loan
loan_amount,"18,700.0000"
term_days,30
interest_rate,0.1000
employment_status,self_employed
declared_monthly_income,"66,240.0000"


## Univariate signal on the training window
A single feature with AUC > 0.85 would be flagged as potential leakage.

In [4]:
from credit_risk.features.leakage import univariate_auc_screen

train = q("SELECT * FROM features.model_dataset WHERE sample_split = 'train'")
univariate_auc_screen(train).head(12)

,feature,coverage,univariate_auc,flag
0,largest_loan,0.6453,0.7063,ok
1,loan_amount,1.0000,0.7039,ok
2,average_loan_amount,0.6453,0.7020,ok
3,collection_rate,0.5982,0.6993,ok
4,avg_dpd,0.5982,0.6972,ok
5,late_payment_ratio,0.6451,0.6970,ok
6,declared_monthly_income,0.9849,0.6947,ok
7,transaction_volume_90d,1.0000,0.6917,ok
8,income_proxy_monthly,1.0000,0.6915,ok
9,transaction_count_90d,1.0000,0.6815,ok


## Leakage audit
The truncation test rebuilds features from data that existed before each cut-off and requires identical values (`reports/leakage_audit.md`).

In [5]:
from credit_risk.features.leakage import default_cutoffs, truncation_test

[
    (r.cutoff, r.loans_compared, r.mismatched_cells, r.passed)
    for r in truncation_test(con, settings, default_cutoffs(con))
]

[(datetime.date(2024, 5, 13), 29, 0, True),
 (datetime.date(2024, 7, 9), 30, 0, True),
 (datetime.date(2025, 5, 23), 29, 0, True),
 (datetime.date(2025, 10, 31), 38, 0, True)]